In [ ]:
!pip install lancedb openai contextgem pyarrow requests

In [ ]:
!pip install google-genai numpy

In [37]:
from google.colab import userdata
import os

# Set the key into your environment variables for both OpenAI and ContextGem
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

In [32]:
import os
import lancedb
import pyarrow as pa
from openai import OpenAI
import requests
import shutil
from contextgem import Document, DocumentLLM, StringConcept

# 1. Initialize Clients
client = OpenAI(api_key=OPENAI_API_KEY)

# 2. Read movie corpus file
movie_corpus_url = "https://raw.githubusercontent.com/Balachandar-Ganesan/GenAIArchitect/refs/heads/main/rajinikanth_movies_list.txt"
response = requests.get(movie_corpus_url)
response.raise_for_status()
movie_corpus = response.text

# 3. Process with distinct, high-recall extraction concepts
doc = Document(raw_text=movie_corpus)
llm = DocumentLLM(model="openai/gpt-4o", api_key=OPENAI_API_KEY)

# Pass 1: Focus specifically on Multiple Characters (Physically separate entities)
role_concept = StringConcept(
    name="Multiple Roles",
    description="Identify EVERY movie where Rajinikanth plays 2 or more distinct physical characters. This includes: 1) Twins/Triple roles (Moondru Mugam), 2) Father/Son (Netrikkan), 3) Robot/Human (Enthiran), and 4) Lookalikes (Billa). Ensure Kochadaiiyaan (Rana/Kochadaiiyaan) is included.",
    add_references=True,
    reference_depth="sentences",
    add_justifications=True,
    justification_depth="comprehensive"
)

# Pass 2: Focus specifically on Identity Deception (Faking an identity/disguise)
deception_concept = StringConcept(
    name="Identity Deception",
    description="Identify EVERY movie where a character uses a disguise, fake identity, or impersonates another. Note: Billa (impersonating a lookalike), Thillu Mullu (fake twin), and Baasha (hidden past identity).",
    add_references=True,
    reference_depth="sentences",
    add_justifications=True,
    justification_depth="comprehensive"
)

doc.add_concepts([role_concept, deception_concept])
doc = llm.extract_all(doc, overwrite_existing=True)

# 4. Reset and Re-index LanceDB
db_path = "./movie_rag_db"
if os.path.exists(db_path): shutil.rmtree(db_path)

db = lancedb.connect(db_path)
schema = pa.schema([
    pa.field("vector", pa.list_(pa.float32(), 1536)),
    pa.field("text", pa.string()),
    pa.field("justification", pa.string()),
    pa.field("category", pa.string())
])

def get_embedding(text):
    return client.embeddings.create(input=[text], model="text-embedding-3-small").data[0].embedding

data_to_insert = []
for c_name in ["Multiple Roles", "Identity Deception"]:
    concept = doc.get_concept_by_name(c_name)
    if concept and concept.extracted_items:
        for item in concept.extracted_items:
            content = f"Category: {c_name} | Movie: {item.value} | Details: {item.justification}"
            data_to_insert.append({
                "vector": get_embedding(content),
                "text": str(item.value),
                "justification": str(item.justification),
                "category": c_name
            })

if data_to_insert:
    table = db.create_table("movie_plots", schema=schema, mode="overwrite")
    table.add(data_to_insert)
    print(f"Success: Indexed {len(data_to_insert)} entries.")

# 5. RAG Engine with categorical separation
def ask_movie_rag(query_str, category_filter):
    query_vector = get_embedding(query_str)
    # Increase limit to ensure all extracted movies are captured
    search_results = table.search(query_vector).where(f"category = '{category_filter}'").limit(25).to_list()

    context = "\n".join([f"[{res['category']}] {res['text']}: {res['justification']}" for res in search_results])

    system_prompt = (
        "Use ONLY the provided context from the movie list text. "
        "List all applicable movies clearly. For overlapping cases like 'Billa', "
        "explicitly state that it fits both categories because it has physically distinct characters "
        "AND one character impersonates the other."
    )

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"CONTEXT:\n{context}\n\nQUESTION: {query_str}"}
        ]
    )
    return response.choices[0].message.content

# 6. Final Validation
print("\n--- Logic-Validated Movie Analysis ---\n")
q1 = "List every movie where Rajinikanth plays MULTIPLE physically distinct characters (e.g. Billa, Enthiran, Moondru Mugam, Kochadaiiyaan)."
q2 = "List every movie involving identity deception, disguises, or faked identities (e.g. Billa, Thillu Mullu, Baasha)."

print(f"Q1: {q1}\nA: {ask_movie_rag(q1, 'Multiple Roles')}\n")
print(f"Q2: {q2}\nA: {ask_movie_rag(q2, 'Identity Deception')}")

[contextgem] 2026-09-03 16:06:45.177 | INFO    | Text is being segmented into paragraphs, as no Paragraph instances were provided...
[contextgem] 2026-09-03 16:06:45.181 | SUCCESS | Process `Document initialization` finished in 0.0 seconds.
[contextgem] 2026-09-03 16:06:45.183 | INFO    | Using model openai/gpt-4o
[contextgem] 2026-09-03 16:06:45.184 | INFO    | API base was not provided. Set `api_base`, if applicable.
[contextgem] 2026-09-03 16:06:45.328 | INFO    | Sentence-level segmentation is requested for concepts: ['Multiple Roles', 'Identity Deception'].
[contextgem] 2026-09-03 16:06:45.329 | INFO    | SaT model will be used for sentence segmentation of the document.
[contextgem] 2026-09-03 16:06:45.330 | INFO    | Paragraphs are being split into sentences...
[contextgem] 2026-09-03 16:06:45.332 | INFO    | Loading SaT model sat-3l-sm...
[contextgem] 2026-09-03 16:06:46.078 | INFO    | SaT model loaded successfully.
[contextgem] 2026-09-03 16:06:49.541 | SUCCESS | Process `Aspe

/usr/local/lib/python3.13/dist-packages/litellm/litellm_core_utils/logging_worker.py:75: RuntimeWarning: coroutine 'Logging.async_success_handler' was never awaited
  self._queue = None


Success: Indexed 9 entries.

--- Logic-Validated Movie Analysis ---

Q1: List every movie where Rajinikanth plays MULTIPLE physically distinct characters (e.g. Billa, Enthiran, Moondru Mugam, Kochadaiiyaan).
A: 1. Kochadaiiyaan
2. Billa
3. Moondru Mugam
4. Enthiran
5. Netrikkan

Q2: List every movie involving identity deception, disguises, or faked identities (e.g. Billa, Thillu Mullu, Baasha).
A: The movies involving identity deception, disguises, or faked identities are:

1. Baasha
2. Billa (fits both categories because it has physically distinct characters AND one character impersonates the other)
3. Thillu Mullu
4. Kochadaiiyaan


In [38]:
import os
import shutil
import requests
import pyarrow as pa
import lancedb
from google import genai

# 1. Initialize Gemini Client
# Assumes you have configured GEMINI_API_KEY in your environment variables
#GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "YOUR_GEMINI_API_KEY")
client = genai.Client(api_key=GEMINI_API_KEY)

# 2. Read movie corpus file
movie_corpus_url = "https://raw.githubusercontent.com/Balachandar-Ganesan/GenAIArchitect/refs/heads/main/rajinikanth_movies_list.txt"
response = requests.get(movie_corpus_url)
response.raise_for_status()
movie_corpus = response.text

# 3. Process with distinct, high-recall extraction concepts
doc = Document(raw_text=movie_corpus)
# Swapped the processing LLM backend to Gemini
llm = DocumentLLM(model="google/gemini-2.5-flash", api_key=GEMINI_API_KEY)

# Pass 1: Focus specifically on Multiple Characters (Physically separate entities)
role_concept = StringConcept(
    name="Multiple Roles",
    description="Identify EVERY movie where Rajinikanth plays 2 or more distinct physical characters. This includes: 1) Twins/Triple roles (Moondru Mugam), 2) Father/Son (Netrikkan), 3) Robot/Human (Enthiran), and 4) Lookalikes (Billa). Ensure Kochadaiiyaan (Rana/Kochadaiiyaan) is included.",
    add_references=True,
    reference_depth="sentences",
    add_justifications=True,
    justification_depth="comprehensive"
)

# Pass 2: Focus specifically on Identity Deception (Faking an identity/disguise)
deception_concept = StringConcept(
    name="Identity Deception",
    description="Identify EVERY movie where a character uses a disguise, fake identity, or impersonates another. Note: Billa (impersonating a lookalike), Thillu Mullu (fake twin), and Baasha (hidden past identity).",
    add_references=True,
    reference_depth="sentences",
    add_justifications=True,
    justification_depth="comprehensive"
)

doc.add_concepts([role_concept, deception_concept])
doc = llm.extract_all(doc, overwrite_existing=True)

# 4. Reset and Re-index LanceDB
db_path = "./movie_rag_db"
if os.path.exists(db_path):
    shutil.rmtree(db_path)

db = lancedb.connect(db_path)

# CRITICAL CRITERIA UPDATE: text-embedding-004 dimensions length is 768 (OpenAI was 1536)
schema = pa.schema([
    pa.field("vector", pa.list_(pa.float32(), 768)),
    pa.field("text", pa.string()),
    pa.field("justification", pa.string()),
    pa.field("category", pa.string())
])

def get_embedding(text):
    """Generates 768-dimension vectors natively with Gemini API."""
    response = client.models.embed_content(
        model="text-embedding-004",
        contents=text
    )
    return response.embeddings.values

data_to_insert = []
for c_name in ["Multiple Roles", "Identity Deception"]:
    concept = doc.get_concept_by_name(c_name)
    if concept and concept.extracted_items:
        for item in concept.extracted_items:
            content = f"Category: {c_name} | Movie: {item.value} | Details: {item.justification}"
            data_to_insert.append({
                "vector": get_embedding(content),
                "text": str(item.value),
                "justification": str(item.justification),
                "category": c_name
            })

if data_to_insert:
    table = db.create_table("movie_plots", schema=schema, mode="overwrite")
    table.add(data_to_insert)
    print(f"Success: Indexed {len(data_to_insert)} entries.")

# 5. RAG Engine with categorical separation
def ask_movie_rag(query_str, category_filter):
    query_vector = get_embedding(query_str)
    # Search LanceDB local records matrix
    search_results = table.search(query_vector).where(f"category = '{category_filter}'").limit(25).to_list()

    context = "\n".join([f"[{res['category']}] {res['text']}: {res['justification']}" for res in search_results])

    system_prompt = (
        "Use ONLY the provided context from the movie list text. "
        "List all applicable movies clearly. For overlapping cases like 'Billa', "
        "explicitly state that it fits both categories because it has physically distinct characters "
        "AND one character impersonates the other."
    )

    # Combined System prompt context instruction block for native text generation syntax
    rag_prompt = f"""
    {system_prompt}

    CONTEXT:
    {context}

    QUESTION: {query_str}
    """

    # Migrated from client.chat.completions.create to Gemini API structure
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=rag_prompt
    )
    return response.text

# 6. Final Validation
print("\n--- Logic-Validated Movie Analysis ---\n")
q1 = "List every movie where Rajinikanth plays MULTIPLE physically distinct characters (e.g. Billa, Enthiran, Moondru Mugam, Kochadaiiyaan)."
q2 = "List every movie involving identity deception, disguises, or faked identities (e.g. Billa, Thillu Mullu, Baasha)."

print(f"Q1: {q1}\nA: {ask_movie_rag(q1, 'Multiple Roles')}\n")
print(f"Q2: {q2}\nA: {ask_movie_rag(q2, 'Identity Deception')}")


[contextgem] 2026-09-03 16:24:38.311 | INFO    | Text is being segmented into paragraphs, as no Paragraph instances were provided...
[contextgem] 2026-09-03 16:24:38.319 | SUCCESS | Process `Document initialization` finished in 0.01 seconds.
[contextgem] 2026-09-03 16:24:38.321 | INFO    | Using model google/gemini-2.5-flash
[contextgem] 2026-09-03 16:24:38.324 | INFO    | API base was not provided. Set `api_base`, if applicable.
[contextgem] 2026-09-03 16:24:38.557 | INFO    | Sentence-level segmentation is requested for concepts: ['Multiple Roles', 'Identity Deception'].
[contextgem] 2026-09-03 16:24:38.558 | INFO    | SaT model will be used for sentence segmentation of the document.
[contextgem] 2026-09-03 16:24:38.560 | INFO    | Paragraphs are being split into sentences...
[contextgem] 2026-09-03 16:24:38.562 | INFO    | Loading SaT model sat-3l-sm...
[contextgem] 2026-09-03 16:24:39.502 | INFO    | SaT model loaded successfully.
[contextgem] 2026-09-03 16:24:42.365 | SUCCESS | Pr

LLMAPIError: Exception occurred while calling LLM API (after 3 retries) - Original error: litellm.BadRequestError: LLM Provider NOT provided. Pass in the LLM provider you are trying to call. You passed model=google/gemini-2.5-flash
 Pass model as E.g. For 'Huggingface' inference endpoints pass in `completion(model='huggingface/starcoder',..)` Learn more: https://docs.litellm.ai/docs/providers LiteLLM Retried: 3 times